In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs('../data', exist_ok=True)

np.random.seed(42)

N = 80000

print("Generating Home & Vehicle Loan dataset...")

# -- Loan type split
loan_type = np.random.choice(['Home','Vehicle'], N, p=[0.65, 0.35])

# -- Borrower profile
age = np.random.randint(23, 65, N)

income_mo = np.random.lognormal(10.5, 0.7, N).round(0)

employment = np.random.choice(
    ['Salaried','Self-Employed','Business'],
    N,
    p=[0.60,0.25,0.15]
)

emp_years = np.random.randint(0, 30, N)

cibil = np.clip(
    np.random.normal(700, 80, N),
    300,
    900
).astype(int)

# -- Loan features
loan_amt = np.where(
    loan_type=='Home',
    np.random.lognormal(14.0, 0.5, N),
    np.random.lognormal(12.5, 0.5, N)
)

loan_amt = loan_amt.round(-3)

prop_val = np.where(
    loan_type=='Home',
    loan_amt / np.random.uniform(0.50, 0.92, N),
    loan_amt / np.random.uniform(0.60, 0.95, N)
)

ltv = (loan_amt / prop_val).round(3)

tenure_mo = np.where(
    loan_type=='Home',
    np.random.choice([120,180,240,300,360], N),
    np.random.choice([12,24,36,48,60,72], N)
)

interest = np.where(
    loan_type=='Home',
    np.random.uniform(7.5, 11.0, N),
    np.random.uniform(8.5, 14.0, N)
).round(2)

# EMI calculation
mo_rate = interest / (12 * 100)

emi = (
    loan_amt * mo_rate * (1+mo_rate)**tenure_mo /
    ((1+mo_rate)**tenure_mo - 1)
).round(0)

emi_income = (emi / income_mo).round(3)

# Additional risk features
delinq_hist = np.random.poisson(0.3, N).clip(0, 5)

num_loans = np.random.randint(0, 6, N)

co_applicant = np.random.binomial(1, 0.45, N)

city_tier = np.random.choice(
    ['Tier1','Tier2','Tier3'],
    N,
    p=[0.35,0.40,0.25]
)

# -- Default probability logic
default_prob = (
    0.04
    + 0.18 * (ltv > 0.85).astype(float)
    + 0.12 * (emi_income > 0.40).astype(float)
    + 0.15 * (cibil < 650).astype(float)
    - 0.08 * (cibil > 750).astype(float)
    + 0.10 * (delinq_hist > 0).astype(float)
    - 0.04 * co_applicant
    + 0.06 * (employment == 'Self-Employed').astype(float)
    + 0.04 * (loan_type == 'Vehicle').astype(float)
    + np.random.normal(0, 0.03, N)
).clip(0.01, 0.90)

defaulted = (
    np.random.uniform(0,1,N) < default_prob
).astype(int)

# -- Time to default
time_to_evt = np.where(
    defaulted == 1,
    np.random.uniform(6, tenure_mo * 0.7, N).clip(1, 360).astype(int),
    np.random.uniform(tenure_mo * 0.5, tenure_mo, N).astype(int)
)

time_to_evt = time_to_evt.clip(1, 360).astype(int)

# -- Final DataFrame
df = pd.DataFrame({
    'loan_id': [f"LN{i:08d}" for i in range(N)],
    'loan_type': loan_type,
    'age': age,
    'income_mo': income_mo.astype(int),
    'employment': employment,
    'emp_years': emp_years,
    'cibil': cibil,
    'loan_amt': loan_amt.astype(int),
    'property_val': prop_val.round(-3).astype(int),
    'ltv': ltv,
    'tenure_mo': tenure_mo,
    'interest_rate': interest,
    'emi': emi.astype(int),
    'emi_income': emi_income,
    'delinq_hist': delinq_hist,
    'num_loans': num_loans,
    'co_applicant': co_applicant,
    'city_tier': city_tier,
    'defaulted': defaulted,
    'time_to_event': time_to_evt,
    'event': defaulted,
})

# Save dataset
df.to_csv('../data/home_vehicle_loans.csv', index=False)

print(f"Shape: {df.shape}")
print(f"Default rate: {defaulted.mean()*100:.1f}%")
print(f"Home loans: {(loan_type=='Home').sum():,}")
print(f"Vehicle loans: {(loan_type=='Vehicle').sum():,}")
print(f"Avg LTV: {ltv.mean():.3f}")
print(f"Avg EMI/Income: {emi_income.mean():.3f}")

print("Saved: data/home_vehicle_loans.csv")

Generating Home & Vehicle Loan dataset...
Shape: (80000, 21)
Default rate: 18.3%
Home loans: 52,031
Vehicle loans: 27,969
Avg LTV: 0.732
Avg EMI/Income: 0.451
Saved: data/home_vehicle_loans.csv
